End-to-End Credit Scoring System

XGBoost · Calibration · Explainability · Production Pipeline

This notebook demonstrates a complete, production-oriented credit scoring system designed for fintech deployment.
It covers data simulation, feature engineering, imbalance handling, model training, calibration, explainability, and packaging for API use.

1. Environment Setup & Imports

In [ ]:
# Install required packages (run once)
!pip install -q scikit-learn==1.7.2 xgboost==1.7.6 imbalanced-learn==0.14.0 shap==0.46.0 joblib==1.4.2 matplotlib seaborn

# Core libraries
import os
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Reproducibility
np.random.seed(42)

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    precision_recall_curve, auc, roc_auc_score,
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, brier_score_loss,
    roc_curve
)
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression

# Imbalanced learning
from imblearn.over_sampling import SMOTE

# XGBoost
from xgboost import XGBClassifier

print("Environment ready")


2. Synthetic Data Generation

Synthetic data is used to simulate realistic borrower behavior when real production data is unavailable.

In [ ]:
N = 50000
age = np.random.randint(18, 65, N)
gender = np.random.choice(["Male", "Female"], N, p=[0.55, 0.45])
income = np.random.normal(150000, 60000, N).clip(20000, 1_000_000)
savings = income * np.random.uniform(0.05, 0.4, N)
loan_amount = np.random.normal(80000, 30000, N).clip(10000, 500000)
loan_term_months = np.random.choice([3, 6, 9, 12, 18, 24], N)
transactions_per_month = np.random.poisson(15, N)
previous_defaults = np.random.poisson(0.2, N)
repayment_ratio = np.random.uniform(0.5, 1.0, N)

debt_to_income = loan_amount / income
linear_score = -3 + 8 * debt_to_income + 0.5 * previous_defaults - 2 * repayment_ratio
prob_default = 1 / (1 + np.exp(-linear_score))
default_flag = np.random.binomial(1, prob_default)

df = pd.DataFrame({
    "age": age,
    "gender": gender,
    "income": income.round(0),
    "savings": savings.round(0),
    "loan_amount": loan_amount.round(0),
    "loan_term_months": loan_term_months,
    "transactions_per_month": transactions_per_month,
    "previous_defaults": previous_defaults,
    "repayment_ratio": repayment_ratio.round(2),
    "debt_to_income": debt_to_income,
    "prob_default": prob_default.round(3),
    "default_flag": default_flag
})

df.shape, df["default_flag"].mean()


3. Exploratory Data Analysis (EDA)

In [ ]:
df.info()
df.describe().T
df.isnull().sum()
df["default_flag"].value_counts(normalize=True)


In [ ]:
sns.countplot(x="default_flag", data=df)
plt.title("Target Distribution")
plt.show()


In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(df.select_dtypes(np.number).corr(), cmap="coolwarm", center=0)
plt.title("Feature Correlation Heatmap")
plt.show()


4. Multicollinearity Check (VIF)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_vif = df.drop(columns=["default_flag", "prob_default"]).copy()
X_vif["gender"] = LabelEncoder().fit_transform(X_vif["gender"])
X_vif["const"] = 1

vif = pd.DataFrame({
    "feature": X_vif.columns,
    "VIF": [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
}).sort_values("VIF", ascending=False)

vif


5. Feature Engineering & Encoding

In [ ]:
df_encoded = df.copy()
df_encoded["gender"] = LabelEncoder().fit_transform(df_encoded["gender"])

df_encoded["installment_per_month"] = df_encoded["loan_amount"] / df_encoded["loan_term_months"]
df_encoded["installment_to_income"] = df_encoded["installment_per_month"] / df_encoded["income"]
df_encoded["savings_to_loan"] = df_encoded["savings"] / (df_encoded["loan_amount"] + 1)

df_encoded = df_encoded.drop(columns=["prob_default"])


6. Scaling & Class Balancing

In [ ]:
X = df_encoded.drop("default_flag", axis=1)
y = df_encoded["default_flag"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

smote = SMOTE(random_state=42)
X_bal, y_bal = smote.fit_resample(X_scaled, y)

np.bincount(y), np.bincount(y_bal)


7. Final Model Training

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal, test_size=0.2, random_state=42
)

model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    use_label_encoder=False,
    eval_metric="logloss"
)

model.fit(X_train, y_train)


8. Threshold Optimization & Evaluation

In [ ]:
y_proba = model.predict_proba(X_test)[:,1]
precision, recall, thresholds = precision_recall_curve(y_test, y_proba)

f1 = 2 * precision * recall / (precision + recall + 1e-6)
opt_idx = np.argmax(f1)
opt_threshold = thresholds[opt_idx]

opt_threshold, precision[opt_idx], recall[opt_idx], f1[opt_idx]


In [ ]:
y_pred = (y_proba >= opt_threshold).astype(int)
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.show()

print(classification_report(y_test, y_pred))


9. Model Calibration (Isotonic Regression)

In [ ]:
X_train_p, X_val, y_train_p, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

model.fit(X_train_p, y_train_p)
val_probs = model.predict_proba(X_val)[:,1]

calibrator = IsotonicRegression(out_of_bounds="clip")
calibrator.fit(val_probs, y_val)

test_probs_raw = model.predict_proba(X_test)[:,1]
test_probs_cal = calibrator.predict(test_probs_raw)

roc_auc_score(y_test, test_probs_cal), brier_score_loss(y_test, test_probs_cal)


In [ ]:
pt, pp = calibration_curve(y_test, test_probs_cal, n_bins=10)
plt.plot(pp, pt, marker="o")
plt.plot([0,1],[0,1],"--")
plt.title("Calibration Curve")
plt.show()


10. Explainability (SHAP)

In [ ]:
import shap

explainer = shap.Explainer(model, X_train)
shap_values = explainer(X_test)

shap.summary_plot(shap_values, X_test, plot_type="bar")
shap.summary_plot(shap_values, X_test)


11. Production-Grade Unified Pipeline

In [2]:
from sklearn.base import BaseEstimator, ClassifierMixin

class CreditScoringPipeline(BaseEstimator, ClassifierMixin):

    def __init__(self, features, threshold, scaler, model, calibrator):
        self.features = features
        self.threshold = threshold
        self.scaler = scaler
        self.model = model
        self.calibrator = calibrator

    def predict_proba(self, X):
        missing = set(self.features) - set(X.columns)
        if missing:
            raise ValueError(f"Missing required features: {missing}")

        X_scaled = self.scaler.transform(X[self.features])
        prob = self.model.predict_proba(X_scaled)[:,1]
        prob = self.calibrator.predict(prob)
        return np.vstack([1 - prob, prob]).T

    def predict(self, X):
        return (self.predict_proba(X)[:,1] >= self.threshold).astype(int)

12. Save Artifacts for API Deployment

In [ ]:
os.makedirs("models", exist_ok=True)

joblib.dump({
    "model": model,
    "scaler": scaler,
    "calibrator": calibrator,
    "threshold": opt_threshold,
    "features": X.columns.tolist()
}, "models/credit_model.pkl")


13. Inference Example (API-Compatible)

In [ ]:
artifact = joblib.load("models/credit_model.pkl")

dummy = pd.DataFrame([np.zeros(len(artifact["features"]))],
                     columns=artifact["features"])

X_scaled = artifact["scaler"].transform(dummy)
prob = artifact["calibrator"].predict(
    artifact["model"].predict_proba(X_scaled)[:,1]
)

prediction = int(prob >= artifact["threshold"])
prob[0], prediction


Model Explainability, Calibration, and Inference Pipeline

This section documents the SHAP explainability workflow, manual isotonic calibration, single and batch inference, and production-ready Streamlit integration for the QuantScore credit scoring system.

The system is fully integrated with the API and uses saved model artifacts for reproducibility.

1. Global SHAP Feature Importance (Magnitude + Direction)

In [ ]:
# --- 1. Extract mean absolute SHAP values (importance)
#     and mean SHAP values (direction)
shap_values_array = shap_values.values  # shape: (n_samples, n_features)
feature_names = X.columns #  Get feature names from the original DataFrame X

mean_abs_shap = np.abs(shap_values_array).mean(axis=0)
mean_shap = shap_values_array.mean(axis=0)

# --- 2. Combine into a DataFrame
shap_summary_df = pd.DataFrame({
    "Feature": feature_names,
    "Mean |SHAP|": mean_abs_shap,
    "Mean SHAP (direction)": mean_shap
})

# --- 3. Sort by importance and select top 10
shap_summary_df = (
    shap_summary_df
    .sort_values(by="Mean |SHAP|", ascending=False)
    .head(10)
)

print("Top 10 Features with SHAP importance and directional effect:")
print(shap_summary_df)


# --- 1. Extract mean absolute SHAP values (importance)
#     and mean SHAP values (direction)
shap_values_array = shap_values.values  # shape: (n_samples, n_features)
feature_names = X_test.columns

mean_abs_shap = np.abs(shap_values_array).mean(axis=0)
mean_shap = shap_values_array.mean(axis=0)

# --- 2. Combine into a DataFrame
shap_summary_df = pd.DataFrame({
    "Feature": feature_names,
    "Mean |SHAP|": mean_abs_shap,
    "Mean SHAP (direction)": mean_shap
})

# --- 3. Sort by importance and select top 10
shap_summary_df = (
    shap_summary_df
    .sort_values(by="Mean |SHAP|", ascending=False)
    .head(10)
)

print("Top 10 Features with SHAP importance and directional effect:")
print(shap_summary_df)


2. SHAP Bar Plot (Magnitude + Direction)

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(
    data=shap_summary_df,
    x="Mean |SHAP|",
    y="Feature",
    hue="Mean SHAP (direction)",
    dodge=False,
    palette="coolwarm"
)
plt.title("Top 10 Feature Importance with SHAP Direction")
plt.show()


3. Persist SHAP Outputs

In [ ]:
import joblib

joblib.dump(shap_values, "shap_values.pkl")
shap_summary_df.to_csv("shap_feature_importance.csv", index=False)


Artifacts saved:

shap_values.pkl

shap_feature_importance.csv

4. Manual Isotonic Calibration of XGBoost

Calibration is performed outside sklearn wrappers to retain full SHAP compatibility.

In [ ]:
import joblib
from sklearn.model_selection import train_test_split
from sklearn.isotonic import IsotonicRegression
import numpy as np
import pandas as pd # Import pandas

# --- 1. Use the pre-trained XGBoost model from earlier steps
raw_model = model # Use the 'model' variable which holds the trained XGBClassifier

# --- 2. Create calibration set (10% of training data)
X_calib, _, y_calib, _ = train_test_split(
    X_train,
    y_train,
    test_size=0.9,
    random_state=42,
    stratify=y_train
)

# --- 3. Scale calibration data
# Convert X_calib back to DataFrame with feature names to avoid UserWarning
X_calib_df = pd.DataFrame(X_calib, columns=X.columns)
X_calib_scaled = scaler.transform(X_calib_df)

# --- 4. Raw probability predictions
y_prob_calib_raw = raw_model.predict_proba(X_calib_scaled)[:, 1]

# --- 5. Fit isotonic regression
isotonic_calibrator = IsotonicRegression(out_of_bounds="clip")
isotonic_calibrator.fit(y_prob_calib_raw, y_calib)

# --- 6. Save calibrated artifacts
joblib.dump(
    {
    "model": raw_model,
    "calibrator": isotonic_calibrator,
    "features": X.columns.tolist(), # Corrected: Use X.columns for feature names
    "scaler": scaler
}, "models/xgb_calibrated.pkl")

5. Sample Prediction Using Calibrated Model

In [ ]:
import joblib
import pandas as pd
import numpy as np

art = joblib.load("models/xgb_calibrated.pkl")

base_model = art["model"]
calibrator = art["calibrator"]
scaler = art["scaler"]
features = art["features"]

sample = {f: 0 for f in features}
sample.update({
    "age": 35,
    "income": 150000,
    "loan_amount": 80000
})

X_sample = pd.DataFrame([sample])[features]
numeric_cols = X_sample.select_dtypes(include="number").columns
X_sample[numeric_cols] = scaler.transform(X_sample[numeric_cols])

raw_prob = base_model.predict_proba(X_sample)[:, 1][0]
calibrated_prob = calibrator.predict([raw_prob])[0]

print("Sample predicted default probability:", round(float(calibrated_prob), 3))


6. Local SHAP Explanation (Single Applicant)

In [ ]:
import shap
import matplotlib.pyplot as plt

explainer = shap.Explainer(base_model, X_train)
shap_values_sample = explainer(X_sample)

plt.title("Local SHAP Explanation for Sample Prediction")
shap.plots.waterfall(shap_values_sample[0], max_display=10, show=True)

7. Unified Applicant Prediction + SHAP Explanation Function

In [4]:
def explain_applicant(applicant_dict):
    art = joblib.load("models/xgb_calibrated.pkl")

    base_model = art["model"]
    calibrator = art["calibrator"]
    scaler = art["scaler"]
    features = art["features"]

    X_sample = pd.DataFrame([applicant_dict])[features]
    numeric_cols = X_sample.select_dtypes(include="number").columns
    X_sample[numeric_cols] = scaler.transform(X_sample[numeric_cols])

    raw_prob = base_model.predict_proba(X_sample)[:, 1][0]
    calibrated_prob = calibrator.predict([raw_prob])[0]

    print(f"Predicted default probability: {calibrated_prob:.3f}")

    explainer = shap.Explainer(base_model, X_train_scaled)
    shap_values = explainer(X_sample)

    shap.plots.waterfall(shap_values[0], max_display=10, show=True)

    return {
        "probability": calibrated_prob,
        "shap_values": shap_values
    }

9. Streamlit Production Integration

The Streamlit app:

Loads calibrated artifacts

Recomputes engineered features

Applies consistent scaling

Uses raw XGBoost for SHAP

Uses isotonic calibrator for probabilities

All logic exactly matches training and inference pipelines.

This ensures:

Probability correctness

Explainability integrity

Deployment safet

10. Model Performance Snapshot

F1-maximizing threshold: 0.407

Precision: 0.786
Recall:    0.823
F1-score:  0.804
Accuracy:  0.80


Final Notes

SHAP always uses the raw XGBoost model

Calibration is applied after prediction

Feature order and scaling are preserved end-to-end